# ⚡ Module 03: Activation Functions

> **Step 4: Deep Learning**  
> **Module: Activation Functions**  
> **Date: 03 June 2026**  
> **Framework: PyTorch & NumPy**  
> **Goal: Master forward passes, gradient calculations, vanishing/exploding gradients, modern activations, and output layer pairings.**

---

## 1. 📌 Overview & Mathematical Formulations
Neural networks compute linear combinations $z = Wx + b$. Activation functions $a = f(z)$ introduce non-linearity, enabling deep networks to act as universal function approximators.

- **Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$
- **Tanh:** $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$
- **ReLU:** $\text{ReLU}(z) = \max(0, z)$
- **Leaky ReLU:** $\text{LeakyReLU}(z) = \max(\alpha z, z)$ (typically $\alpha = 0.01$)
- **ELU:** $\text{ELU}(z) = \begin{cases} z & z > 0 \\ \alpha(e^z - 1) & z \le 0 \end{cases}$
- **GELU:** $\text{GELU}(z) \approx 0.5z \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \left(z + 0.044715 z^3\right)\right)\right)$
- **SiLU (Swish):** $\text{SiLU}(z) = z \cdot \sigma(z)$
- **Softmax:** $\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$

## 2. 🧮 Pure NumPy Implementations & Exact Derivatives

In [ ]:
import numpy as np

# 1. Sigmoid & Derivative
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

# 2. Tanh & Derivative
def tanh(z):
    return np.tanh(z)

def tanh_grad(z):
    return 1 - np.tanh(z)**2

# 3. ReLU & Derivative
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return np.where(z > 0, 1.0, 0.0)

# 4. Leaky ReLU & Derivative
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_grad(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

# Verification on test inputs
test_z = np.array([-4.0, -2.0, 0.0, 2.0, 7.0])
print("Inputs:    ", test_z)
print("Sigmoid:   ", np.round(sigmoid(test_z), 4))
print("ReLU:      ", relu(test_z))
print("Leaky ReLU:", leaky_relu(test_z))

## 3. 📈 Visualizing Activations, Derivatives, & Vanishing Gradients

In [ ]:
import matplotlib.pyplot as plt

z = np.linspace(-5, 5, 500)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Sigmoid Plot
axes[0, 0].plot(z, sigmoid(z), label="Sigmoid", color="blue")
axes[0, 0].plot(z, sigmoid_grad(z), label="Derivative (max 0.25)", color="blue", linestyle="--")
axes[0, 0].set_title("Sigmoid (Saturates for |z| > 3)")
axes[0, 0].grid(True); axes[0, 0].legend()

# Tanh Plot
axes[0, 1].plot(z, tanh(z), label="Tanh", color="green")
axes[0, 1].plot(z, tanh_grad(z), label="Derivative (max 1.0)", color="green", linestyle="--")
axes[0, 1].set_title("Tanh (Zero-Centered Output)")
axes[0, 1].grid(True); axes[0, 1].legend()

# ReLU Plot
axes[1, 0].plot(z, relu(z), label="ReLU", color="red")
axes[1, 0].plot(z, relu_grad(z), label="Derivative", color="red", linestyle="--")
axes[1, 0].set_title("ReLU (Zero Gradient for z < 0)")
axes[1, 0].grid(True); axes[1, 0].legend()

# Leaky ReLU Plot
axes[1, 1].plot(z, leaky_relu(z), label="Leaky ReLU", color="purple")
axes[1, 1].plot(z, leaky_relu_grad(z), label="Derivative (0.01 slope)", color="purple", linestyle="--")
axes[1, 1].set_title("Leaky ReLU (Prevents Dying Neurons)")
axes[1, 1].grid(True); axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. 🧪 Modern Activations in PyTorch (GELU, SiLU) & Multi-Class Softmax

In [ ]:
import torch
import torch.nn as nn

# 1. Softmax Numerically Stable Implementation
def softmax_np(logits):
    exp_l = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return exp_l / np.sum(exp_l, axis=-1, keepdims=True)

logits = np.array([2.0, 1.0, 0.1])
probs = softmax_np(logits)
print("Softmax Probabilities:", np.round(probs, 4))
print("Sum of Probabilities: ", np.sum(probs))

# 2. Modern PyTorch Hidden Activations
x_tensor = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
gelu = nn.GELU()
silu = nn.SiLU()
prelu = nn.PReLU()

print("\nPyTorch GELU: ", gelu(x_tensor).numpy())
print("PyTorch SiLU: ", silu(x_tensor).numpy())
print("PyTorch PReLU:", prelu(x_tensor).detach().numpy())

## 5. 🎯 Decision Matrix: Output Layer & Loss Selection

| Problem Type | Output Activation | Loss Function in PyTorch | Target Shape |
| :--- | :--- | :--- | :--- |
| **Regression** | None (Linear) | `nn.MSELoss()` or `nn.L1Loss()` | `(N, 1)` or `(N, K)` |
| **Binary Classification** | None (Raw Logit) | `nn.BCEWithLogitsLoss()` | `(N, 1)` |
| **Multi-Class Classification** | None (Raw Logits) | `nn.CrossEntropyLoss()` | `(N, C)` |
| **Multi-Label Classification** | None (Raw Logits) | `nn.BCEWithLogitsLoss()` | `(N, C)` |

## 6. 📝 Exercises & Solutions

### Exercise 1 Solutions:
- $\text{ReLU}(-4) = \max(0, -4) = 0$
- $\text{ReLU}(7) = \max(0, 7) = 7$

### Exercise 2 Solutions:
- $\sigma(0) = \frac{1}{1 + e^0} = 0.5$
- $\sigma(2) = \frac{1}{1 + e^{-2}} \approx 0.8808$

### Exercise 3 Solution:
Sequential linear layers without activations multiply matrices directly:  
$$y = W_3(W_2(W_1 x + b_1) + b_2) + b_3 = (W_3 W_2 W_1)x + (W_3 W_2 b_1 + W_3 b_2 + b_3) = W_{net} x + b_{net}$$
The network collapses into a single linear transformation regardless of depth.

### Exercise 5 Output Setup Solutions:
- **House Price:** Regression $\to$ Identity Output + `nn.MSELoss()`
- **Spam / Not Spam:** Binary Classification $\to$ Single Logit Output + `nn.BCEWithLogitsLoss()`
- **Cat / Dog / Horse:** Multi-class Classification $\to$ 3 Logits Output + `nn.CrossEntropyLoss()`